In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import platform
import os
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def setup_korean_font():
    # 마이너스 기호 깨짐 방지
    mpl.rcParams['axes.unicode_minus'] = False

    system = platform.system()
    if system == "Darwin":  # macOS
        preferred = "AppleGothic"
    elif system == "Windows":
        preferred = "Malgun Gothic"
    else:  # Linux
        preferred = "NanumGothic"  # 설치 필요: 예) Ubuntu: sudo apt install fonts-nanum

    # 설치된 폰트 리스트
    available = {f.name for f in fm.fontManager.ttflist}

    if preferred in available:
        mpl.rcParams["font.family"] = preferred
    else:
        # fallback: 사용자에게 유사한 한글 폰트 제안
        candidates = sorted([f for f in available if any(k in f.lower() for k in ["nanum", "gothic", "malgun", "apple"])])
        print(f"'{preferred}' 폰트를 찾지 못했습니다. 대체 가능한 한글계열 폰트 예시: {candidates[:5]}")
        if candidates:
            mpl.rcParams["font.family"] = candidates[0]
        else:
            print("한글 폰트가 없으면 명시적으로 ttf 파일을 내려받아 지정하세요.")

# 로컬 TTF를 직접 사용할 경우 (예: NanumGothic 다운로드해서 경로 지정)
def use_local_font(ttf_path):
    if os.path.isfile(ttf_path):
        fm.fontManager.addfont(ttf_path)
        prop = fm.FontProperties(fname=ttf_path)
        name = prop.get_name()
        mpl.rcParams["font.family"] = name
        mpl.rcParams['axes.unicode_minus'] = False
    else:
        raise FileNotFoundError(f"{ttf_path} 이 존재하지 않습니다.")

# 설정 실행
setup_korean_font()

In [ ]:
import pandas as pd 
# phase1
df1 = pd.read_csv('/Users/qwergs/Desktop/필드/phase1/가구원_데이터셋.csv')
df2 = pd.read_excel('/Users/qwergs/Desktop/필드/phase1/가구원_변수.xlsx')
hm_code = pd.read_csv('/Users/qwergs/Desktop/필드/phase1/가구원_코드북.csv')

In [ ]:
# phase2 
btc = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/비트코인.csv')
eth = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/이더리움.csv')
bond_kor = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/채권.csv')
# etf = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/국내ETF.csv', encoding = enc)
# kospi_tpo100 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv')

- 파일 인코딩 자동 탐지

In [ ]:
import chardet

# 파일 앞부분만 읽어서 인코딩 추측
raw = open('/Users/qwergs/Desktop/필드/phase2/국내ETF.csv', 'rb').read(10000)
enc = chardet.detect(raw)['encoding']
print("추측된 인코딩:", enc)

# 추측된 인코딩으로 다시 읽기
etf = pd.read_csv(
    '/Users/qwergs/Desktop/필드/phase2/국내ETF.csv',
    encoding=enc
)

In [ ]:
import chardet

# 파일 앞부분 바이트를 읽어서 인코딩 추측
raw = open('/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv','rb').read(5000)
enc = chardet.detect(raw)['encoding']
print("추측 인코딩:", enc)

# 추측된 인코딩으로 다시 읽기
kospi_tpo100 = pd.read_csv(
    '/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv',
    encoding=enc
)

In [ ]:
df1.columns

In [ ]:
hm_dataset = df1.copy()
hm_var = df2.copy()

In [ ]:
hm_dataset.info()

In [ ]:
hm_dataset.columns

#### 가구원 데이터 셋의 컬럼과 코드북에서 주어진 컬럼의 정보를 합쳐 멀티인덱스로 만드는 과정
- 계속 코드북과 대조하며 보는 불편함을 해소하기 위한 작업

In [ ]:
# 변수 시트 
cb_path = '/Users/qwergs/Desktop/필드/phase1/가구원_변수.xlsx'
codebook = pd.read_excel(cb_path,
                         sheet_name="변수 시트",
                         usecols = ['변수명', '변수 설명'])
var2desc = dict(zip(codebook['변수명'], codebook['변수 설명']))

In [ ]:
print(var2desc)

In [ ]:
codebook.head

In [ ]:
orig_cols = list(hm_dataset.columns)

In [ ]:
tuples = [
    (col, var2desc.get(col, ""))
    for col in orig_cols
]

In [ ]:
multi_idx = pd.MultiIndex.from_tuples(
    tuples,
    names = ['variable', 'description']
)

In [ ]:
hm_dataset.columns = multi_idx
hm_dataset.columns[:5]

In [ ]:
display(hm_dataset.isna().sum())

In [ ]:
hm_dataset.head(20)

In [ ]:
hm_dataset.info()

In [ ]:
df = hm_dataset.copy()
df.columns = df.columns.get_level_values("variable")

In [ ]:
# 소비 비율 
df['p16aa123'] = df.apply(
    lambda row: (row['p16bd002'] / row['p_16_inc_all']) * 100 
    if row['p_16_inc_all'] != 0 else None,
    axis=1
)

In [ ]:
# 보험 가입 개수 열 추가 
# 보험 가입 항목이 1인 경우만 세기
col_to_sum = ['p16bc030','p16bc044', 'p16bc046', 'p16bc048', 'p16bc050', 'p16bc052', 'p16bc054']

# 각 열에서 값이 1인 경우만 True → True=1로 간주하고 행별 합산
df['p16bc111'] = (df[col_to_sum] == 1).sum(axis=1)

# 원본 열 삭제
df.drop(columns=col_to_sum, inplace=True)

In [ ]:
df.columns

In [ ]:
df = df.drop('p16bc007', axis = 1)
df = df.drop('p16bc061', axis = 1)
df = df.drop('p16bc021', axis = 1)

## Phase1. 데이터 군집 분석 및 행동경제학적 페르소나 모델링 
- task 1.1. 주어진 가구원 데이터를 적절한 기법을 사용하여 전처리 및 군집화를 진행하고 경과 해석
- task 1.2. 위의 군집화 결과를 바탕으로, 행동경제학적 특징을 고려하여 페르소나 구축

`고려사항 : 데이터 전처리 및 군집화 과정에서 사용한 기법데 대해 설명`

In [ ]:
# 금액 관련 컬럼 
money_cols = [
    'p16bd002','p16bd008','p16bd006','p16bd010','p16bd007','p16bc051','p16bc026','p_16_inc_all',
    'p16bd012','p16bd018','p16bd015','p16ge041','p16ge042','p16aa123'
]
# 결측값 선형 보간법으로 채우기
df[money_cols] = df[money_cols].interpolate(method='linear', axis=0)

### **금액관련 변수 선형 보간법으로 결측치 처리**

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
na_summary = df.isna().sum()
na_columns_with_count = na_summary[na_summary > 0]
print(na_columns_with_count)

In [ ]:
df['p16aa009'] = df['p16aa009'].fillna(-9)

In [ ]:
# 결측치가 하나라도 있는 행 제거 (해당 컬럼에 한해)
cols_to_check = [
    'p16gk001', 'p16gk002', 'p16gk003', 'p16gk004', 'p16gb001', 'p16gb020',
    'p16gb030', 'p16gb121', 'p16gb122', 'p16gb123', 'p16gb124',
    'p16gb050', 'p16gb060', 'p16gb080', 'p16gb090', 'p16gb110',
    'p16ge001', 'p16ge020', 'p16ge030'
]

df_cleaned = df.dropna(subset=cols_to_check)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.shape

In [ ]:
df00 = df_cleaned.copy()

In [ ]:
df00.columns

In [ ]:
df_cleaned.isna().sum()

In [ ]:
df_cleaned.columns

### **컬럼 선탹 기준 무엇인가?**
1. 결측치가 많은 컬럼 제거 전체 데이터의 1/3 이상이 펼측치라면 제거 
2. 카테고리별 대표 변수 선별 (같은 카테고리 내에 범주형)
3. 다중공선성 / 상관관계 분석 (동일 카테고리 내 변수들의 관계가 어떤지 확인 후 선별)
4. 회귀분석 (모델 : Lasso, statsmodels(p-value > 0.05이면 유효하지 않음), RFR(재귀적 특징 제거))
    - 목적 : 
        - 설명력이 높은 변수만 선별 
        - 과적합 방지
        - 모델 해석력 향상 
        - 다중공선성 제거

## `Lasso`

In [ ]:
# 회귀분석 
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import pandas as pd

# X, y 정의
X = df00.drop(columns=['p_16_inc_all'])  # 예: 연간소득을 종속변수로 예시
y = df00['p_16_inc_all']

# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
1
# LassoCV로 알파 최적화 + 변수 선택
lasso = LassoCV(cv=5, random_state=42).fit(X_scaled, y)

# 결과 확인
selected_features = X.columns[lasso.coef_ != 0].tolist()
print("선택된 변수:", selected_features)

## `statsmodels (p-value)`

In [ ]:
import statsmodels.api as sm

X = df00.drop(columns=['p_16_inc_all'])
y = df00['p_16_inc_all']

# 상수항 추가
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

## `RFE(재귀적 특징 제거)`

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

model = LinearRegression()
rfe = RFE(model, n_features_to_select=5)
rfe.fit(X_scaled, y)

# 선택된 변수
selected_features = X.columns[rfe.support_].tolist()
print("선택된 변수:", selected_features)

In [ ]:
# col_sel1 = [
#     'p16aa001',  # 평소 소득활동 여부 (주관/객관)
#     'p16aa009',  # * 직업 유형 또는 직종 코드 
#     'p16aa123',  # 소비비율
#     'p16bc001',  # 건강보험 자격 유형
#     'p16bc023',  # 건강보험료 납부 여부
#     'p16bc026',  # 건강보험료 납부 금액 또는 납부 수준
#     'p16bc111',  # 총 보험 가입 수 (민간 + 공공 포함)
#     'p16bd002',  # 신용/직불카드 총 사용액
#     'p16bd004',  # 현금영수증 총 사용 금액
#     'p16bd008',  # 신용카드 사용액
#     'p16gb001',  # 세금 납부 태도 또는 직접 납부 여부
#     'p16gb020',  # 세금 및 정부혜택에 대한 인식/태도
#     'p16gb110',  # 금융 관련 태도 
#     'p16ge001',  # 건강 상태 (주관적 평가)
#     'p16ge041',  # 최소 노후 생활비 월평균
#     'p16ge042'   # 적정 노후 생활비 월평균
# ]

In [ ]:
col_sel1 = ['p16aa123',
    'p_16_inc_all',     # Ba: 가구원 연간소득
    'p16bc111',               # Bc: 보험 가입수 
    'p16bd002', 'p16bd008',   # Bd: 신용카드 사용금액
    'p16gk004',               # Gk: 행복도, 삶의 만족도
    'p16ge001', 'p16ge030',   # Ge: 건강상태, 은퇴 상태 
    'p16aa001', 'p16bc023', 'p16bd004', # 평소 소득활동 여부, 건보료 납부 여부, 사용한 현금 영수증 총 금액
    'p16aa009'                          # 직업 유형
]

In [ ]:
df_sel = df_cleaned[col_sel1].copy()

In [ ]:
df_sel.columns

# 상관관계/회귀/다중공선선 

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled1 = scaler.fit_transform(df_cleaned)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled1 = scaler.fit_transform(df_cleaned)
df_scaled = pd.DataFrame(X_scaled1, columns=df_cleaned.columns)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["변수"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(add_constant(df).values, i+1)
                       for i in range(len(df.columns))]
    return vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)

In [ ]:
def drop_high_vif_features(df, threshold=10.0):
    variables = df.columns.tolist()
    
    while True:
        vif = [variance_inflation_factor(add_constant(df[variables]).values, i+1)
               for i in range(len(variables))]
        
        max_vif = max(vif)
        if max_vif > threshold:
            remove_var = variables[vif.index(max_vif)]
            print(f"VIF {max_vif:.2f} → '{remove_var}' 제거")
            variables.remove(remove_var)
        else:
            break
            
    return df[variables]

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import pandas as pd

def drop_high_vif_features_with_log(df_original, threshold=10.0):
    """
    다중공선성(VIF)이 높은 변수를 자동으로 제거하고, 제거된 변수명을 기록하여 반환합니다.

    Parameters:
        df_original (pd.DataFrame): 원본 수치형 데이터프레임
        threshold (float): 제거 기준이 되는 VIF 임계값

    Returns:
        df_result (pd.DataFrame): 다중공선성 높은 변수를 제거한 원본 df
        removed_vars (list): 제거된 변수 리스트
        vif_table (pd.DataFrame): 최종 남은 변수들의 VIF 테이블
    """
    df = df_original.copy()
    removed_vars = []
    variables = df.columns.tolist()

    while True:
        X = add_constant(df[variables])
        vif = [variance_inflation_factor(X.values, i + 1) for i in range(len(variables))]
        max_vif = max(vif)
        
        if max_vif > threshold:
            max_var = variables[vif.index(max_vif)]
            print(f"VIF {max_vif:.2f} → '{max_var}' 제거")
            removed_vars.append(max_var)
            variables.remove(max_var)
        else:
            break

    # 최종 VIF 계산 결과
    final_X = add_constant(df[variables])
    final_vif = [variance_inflation_factor(final_X.values, i + 1) for i in range(len(variables))]
    vif_table = pd.DataFrame({'변수': variables, 'VIF': final_vif})

    # 제거된 변수만 df_original에서 제거한 최종 결과
    df_result = df_original.drop(columns=removed_vars)

    return df_result, removed_vars, vif_table

In [ ]:
# 1. 스케일링
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled1 = scaler.fit_transform(df_sel)
df_scaled = pd.DataFrame(X_scaled1, columns=df_sel.columns)

# 2. 다중공선성 제거 + 변수 기록
df_result, removed_vars, vif_final = drop_high_vif_features_with_log(df_scaled, threshold=10)

# 3. 원본에서 제거된 변수 확인
print("제거된 변수 목록:", removed_vars)

# 4. 최종 VIF 확인
print(vif_final)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# 카테고리별 대표 변수 
selected_columns = [
    # 'p_16_inc_all',     # Ba: 가구원 연간소득
    # 'p16bc111',               # Bc: 보험 가입수 
    # 'p16bd002', 'p16bd008',   # Bd: 신용카드 사용금액
    # 'p16gk004',               # Gk: 행복도, 삶의 만족도
    # 'p16ge001', 'p16ge030'    # Ge: 건강상태, 적정 노후생활비
    'p16aa123',    # 소비 비율 
    'p_16_inc_all',     # Ba: 가구원 연간소  득
    'p16bc111',   # Bc: 보험 가입수 
    'p16bd002',   # 신용/착불 카드 사용 금액 
    'p16bd008',   # 신용카드 사용금액
    'p16gk004',   # 행복도, 삶의 만족도
    'p16ge001',   # 건강상태
    'p16ge030',   # 은퇴 상태 
    'p16aa001',   # 평소 소득활동 여부
    'p16bc023',   # 건보료 납부 여부
    'p16bd004'    # 사용한 현금 영수증 총 금액
]

In [ ]:
# 대표 변수만 선택
df_sel = df_cleaned[selected_columns].copy()

In [ ]:
# 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sel)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

def find_optimal_k(X, k_range=range(2, 11), random_state=42):
    inertia_list = []
    silhouette_list = []
    db_list = []

    for k in k_range:
        model = KMeans(n_clusters=k, random_state=random_state)
        labels = model.fit_predict(X)
        inertia_list.append(model.inertia_)  # WCSS
        silhouette_list.append(silhouette_score(X, labels))
        db_list.append(davies_bouldin_score(X, labels))

    # 결과 시각화
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))

    ax[0].plot(k_range, inertia_list, marker='o')
    ax[0].set_title('Elbow Method (Inertia)')
    ax[0].set_xlabel('Number of clusters (k)')
    ax[0].set_ylabel('WCSS')

    ax[1].plot(k_range, silhouette_list, marker='o')
    ax[1].set_title('Silhouette Score')
    ax[1].set_xlabel('Number of clusters (k)')
    ax[1].set_ylabel('Silhouette')

    ax[2].plot(k_range, db_list, marker='o')
    ax[2].set_title('Davies-Bouldin Score')
    ax[2].set_xlabel('Number of clusters (k)')
    ax[2].set_ylabel('DB Score (lower is better)')

    plt.tight_layout()
    plt.show()

    # 최적값 추천
    best_silhouette_k = k_range[np.argmax(silhouette_list)]
    best_db_k = k_range[np.argmin(db_list)]

    print(f"Silhouette 기준 최적 k: {best_silhouette_k}")
    print(f"Davies-Bouldin 기준 최적 k: {best_db_k}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# 데이터 전처리
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sel)  # X는 클러스터링에 사용할 feature set

# 최적 k 찾기
find_optimal_k(X_scaled, k_range=range(2, 11))

`Elbow Method – 이론적 근거 부족, 명확한 기준 없음`
- Elbow Method는 K값에 따라 SSE(오차 제곱합) 변화를 시각화하여 “팔꿈치”처럼 꺾이는 지점을 찾는 방식이지만, 이론적 근거가 부족하며 주관적 판단에 의존함.
- SIGKDD Explorations에 발표된 논문에서는 *“Elbow 방법은 그만 써야 한다”*며, *“이 방법은 이론적 근거가 매우 약하다”*고 비판함.
- 실제 사례(당뇨병 환자 클러스터링 등)에서도 Elbow Plot이 명확한 꺾임점을 제공하지 못해 클러스터 수 결정에 실패.
- 결론: Elbow Method는 시각적으로 유용하지만, 명확한 수학적 기준 없이 직관에 의존하므로 절대적 기준으로 쓸 수 없음.

---

`Silhouette Score – 데이터 분포와 형태에 민감함`
- Silhouette Score는 군집 내 응집도와 다른 군집과의 분리를 비교하는 지표로, 1에 가까울수록 좋음.
- 그러나 데이터 형태가 비정형이거나 군집 크기/밀도가 다를 경우 신뢰도 저하.
- 인간 자세 데이터를 활용한 2023년 실험에서는 Silhouette Score가 너무 적은 수의 클러스터(3개)를 추천해, 실제 구분 가능한 자세들을 놓침.
- 이유는 Silhouette이 큰 간격 차이만 강조하고, 점진적인 차이를 무시하기 때문.
- 결론: Silhouette은 구형/잘 분리된 군집에 적합, 복잡하거나 연속적인 데이터에서는 잘못된 결과를 유도할 수 있음.

---

`Davies–Bouldin Index – 노이즈와 이상치에 민감`
- DBI는 각 군집의 유사도(내부 분산/군집 간 거리 비율)를 기반으로 하며, 값이 낮을수록 좋음.
- 하지만 DBI는 노이즈나 군집 모양의 비구형성에 매우 민감.
- 위 자세 데이터 사례에서는 DBI가 지나치게 많은 클러스터(5개)를 추천, 같은 동작이 상황에 따라 다르게 인식되며 과도한 분할 발생.
- DBI는 군집 내 분산이 낮은 걸 선호하여, 실제 동일한 패턴도 노이즈로 인해 분리할 위험이 있음.
- 결론: DBI는 구형/분산이 작은 군집에 적합, 복잡하고 노이즈가 많은 데이터에는 과적합 유발 가능.

---

`실험적 사례: 지표 간 불일치 및 데이터 의존성`
- 여러 실험 논문에서 지표마다 추천하는 클러스터 수(k)가 일치하지 않음.
- 자세 분류 연구: Silhouette은 3개, DBI와 Dunn Index는 5개, 다른 지표는 6개를 추천 → 결국 타협안으로 4개 선택.
- 야구 선수 데이터 등에서도 각 지표가 서로 다른 결과를 냄.
- 당뇨병 환자 클러스터링 연구: Elbow, Silhouette, Gap Statistic을 사용했지만 모두 명확한 클러스터 수를 제시하지 못함.
- 결론: 지표들은 각기 다른 기준을 반영하므로 하나의 지표만으로 클러스터 수를 결정하는 것은 부적절.


## *참고 문헌*

> Karim, Iliyas & Daud, Hanita & Zainuddin, Nooraini & Sokkalingam, Rajalingam & Farooq, Muhammad & Baig, Elahi & Ayub, Gohar & Zafar, Mudasar. (2024). Determining the optimal number of clusters by Enhanced Gap Statistic in K-mean algorithm. Egyptian Informatics Journal. 27. 100504. 10.1016/j.eij.2024.100504. 


> Karim, Iliyas & Daud, Hanita & Zainuddin, Nooraini & Sokkalingam, Rajalingam & Farooq, Muhammad & Baig, Elahi & Ayub, Gohar & Zafar, Mudasar. (2024). Determining the optimal number of clusters by Enhanced Gap Statistic in K-mean algorithm. Egyptian Informatics Journal. 27. 100504. 10.1016/j.eij.2024.100504.  


> Khan, I. K., Daud, H. B., Zainuddin, N. B., Sokkalingam, R., Farooq, M., Baig, M. E., Ayub, G., & Zafar, M. (2024).
Determining the optimal number of clusters by Enhanced Gap Statistic in K-mean algorithm.
Egyptian Informatics Journal, 27, 100504. https://doi.org/10.1016/j.eij.2024.100504


> Lugner, M., Gudbjörnsdottir, S., Sattar, N., Svensson, A.-M., Miftaraj, M., Eeg-Olofsson, K., Eliasson, B., & Franzén, S. (2021).
Comparison between data-driven clusters and models based on clinical features to predict outcomes in type 2 diabetes: nationwide observational study.
Diabetologia, 64, 1973–1981. https://doi.org/10.1007/s00125-021-05485-5


> Schubert, E. (2021).
Stop using the elbow criterion for k-means and how to choose the number of clusters instead.
SIGKDD Explorations, 23(2), Article 36. https://arxiv.org/abs/2109.10978 (또는 PDF 내부에 명시된 TU Dortmund Technical Letter)




Lugner, M., Gudbjörnsdottir, S., Sattar, N., Svensson, A.-M., Miftaraj, M., Eeg-Olofsson, K., Eliasson, B., & Franzén, S. (2021).Comparison between data-driven clusters and models based on clinical features to predict outcomes in type 2 diabetes: Nationwide observational study. Diabetologia, 64, 1973–1981. https://doi.org/10.1007/s00125-021-05485-5

Schubert, E. (2021). Stop using the elbow criterion for k-means and how to choose the number of clusters instead. SIGKDD Explorations, 23(2), Article 36. https://arxiv.org/abs/2109.10978

In [ ]:
# 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sel)

# KMeans 클러스터링 
kmeans = KMeans(n_clusters=5, random_state=42)
df_sel['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
# PCA 2차원 변환
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

df_sel['PC1'] = components[:, 0]
df_sel['PC2'] = components[:, 1]

# 시각화
plt.figure(figsize=(18, 16))
sns.scatterplot(data=df_sel, x='PC1', y='PC2', hue='cluster', palette='Set2', s=60)
plt.title("클러스터링 결과 (PCA 2D 시각화)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.grid(True)
plt.show()

In [ ]:
# 클러스터별 변수 평균 확인
cluster_summary = df_sel.groupby('cluster').mean(numeric_only=True).round(3)

# 보기 좋게 transpose
cluster_summary.T

# 클러스터 기반 카드 소비 성향 분석 및 행동경제학적 페르소나 도출

본 분석은 주요 소비, 소득, 건강, 은퇴 상태 등의 지표를 바탕으로 KMeans 클러스터링을 수행한 결과이다.  
각 클러스터의 특성을 정량적으로 분석하고, 행동경제학적 관점에서 소비자 유형(페르소나)을 도출하였다.

---

## 사용된 주요 변수 및 해석 기준

| 변수명        | 설명 |
|---------------|------|
| p16aa123      | 소비 비율 (100 초과 시 적자, 이하 시 흑자) |
| p_16_inc_all  | 가구원 연간소득 (만원 단위) |
| p16bc111      | 보험 가입 수 |
| p16bd002      | 신용/직불카드 총 사용 금액 |
| p16bd008      | 신용카드 사용 금액 |
| p16gk004      | 삶의 만족도 (0~10) |
| p16ge001      | 건강상태 (1 = 건강함, 5 = 건강하지 않음) |
| p16ge030      | 은퇴 여부 (1 = 은퇴, 2 = 비은퇴) |
| p16aa001      | 소득활동 여부 (1 = 활동, 2 = 무활동) |
| p16bc023      | 건강보험료 납부 여부 (1 = 납부, 2 = 미납) |
| p16bd004      | 현금영수증 총 사용 금액 |

---

## 클러스터별 변수 평균 요약 (반올림 수치 기준)

| 변수명        | C0  | C1  | C2  | C3  | C4  |
|---------------|-----|-----|-----|-----|-----|
| 소비비율      | 95  | 92  | 71  | 288 | 59  |
| 연간소득      | 1513| 1071| 3604| 8703| 7123|
| 보험가입수    | 1   | 1   | 2   | 2   | 3   |
| 카드사용액    | 886 | 928 |1246 |6136 |2993|
| 신용카드액    | 987 |1028 |1084 |5435 |2374|
| 만족도        | 7   | 6   | 7   | 7   | 7   |
| 건강상태      | 2   | 3   | 2   | 2   | 2   |
| 은퇴 여부     | 2   | 1   | 2   | 2   | 2   |
| 소득활동      | 1   | 2   | 1   | 1   | 1   |
| 건보료납부    | 2   | 2   | 1   | 1   | 1   |
| 현금영수증    | 14  | 4   | 77  |382  |511 |

---

## 클러스터별 해석 및 페르소나 정의

### Cluster 0 - 균형적 소비를 추구하는 근로자

- 중소득, 소비비율 95%로 절제된 지출
- 카드와 신용카드 사용 균형
- 건강보험료는 미납, 현금영수증 사용 적음
- 비은퇴자이며, 소득활동 활발

**페르소나 정의**:  
절제된 소비와 안정적 소득을 유지하는 중소득 근로자. 세제 혜택 및 보험 등 사회보장 참여율은 낮음

---

### Cluster 1 - 은퇴 후 불균형 소비자

- 최저 수준의 소득, 소비비율 92%
- 은퇴자 중심
- 건강상태 가장 나쁘고 만족도도 낮음
- 카드 사용은 유지되나 현금영수증 사용 적음

**페르소나 정의**:  
소득 부족과 건강 문제를 동시에 가진 은퇴 고위험군. 기본적인 소비는 유지하지만 재정 안정성 취약

---

### Cluster 2 - 실속형 절제 소비자

- 중소득, 소비비율 71%로 흑자 구조
- 건강보험료 납부, 현금영수증 적극 사용
- 만족도와 건강 수준 양호
- 소득활동 활발, 비은퇴자 중심

**페르소나 정의**:  
재정적 계획이 명확하고 세금 감면도 고려하는 실속 소비자. 체계적 소비 습관과 균형된 생활

---

### Cluster 3 - 고소득 고소비 금융 의존층

- 최고 소득, 소비비율 288%로 과소비 성향
- 카드, 신용카드 사용량 매우 높음
- 건강상태와 만족도는 양호
- 비은퇴자, 건강보험 납부율 높음

**페르소나 정의**:  
높은 소득을 바탕으로 소비를 극대화하는 유형. 신용 활용도가 높고 미래 재정 리스크 가능성 존재

---

### Cluster 4 - 절제형 고소득 소비자

- 고소득, 소비비율 59%로 가장 절제됨
- 보험 가입 수와 건강보험 납부율 모두 최고
- 현금영수증 사용률도 가장 높음
- 삶의 만족도와 건강 상태도 양호

**페르소나 정의**:  
이상적인 재정 균형을 실현한 모범 소비자. 소득, 소비, 건강관리 및 세제 대응 모두 우수

---

## 종합 요약

| 클러스터 | 페르소나 요약                          | 재정 및 소비 특징               |
|----------|---------------------------------------|-------------------------------|
| C0       | 절제적 중소득 근로자형               | 소비-소득 균형, 보장제도 참여 낮음 |
| C1       | 은퇴 후 재정 불안정형                 | 저소득, 건강 악화, 소비 유지     |
| C2       | 실속 소비형                            | 흑자 구조, 세제 활용 우수       |
| C3       | 과소비형 고소득층                     | 카드 중심 소비 확대, 신용 의존   |
| C4       | 절제형 고소득 최적자                  | 고소득이지만 지출 억제, 재정 안정 |

---